<a href="https://colab.research.google.com/github/JunJul/Technology-News-Insight-Engine/blob/Master/Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install llama-index transformers accelerate bitsandbytes pypdf
!pip install llama-index-llms-huggingface
!pip install langchain
!pip install -U langchain-community
!pip install llama-index-embeddings-langchain

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
import nltk
nltk.download('vader_lexicon')

In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/Technology News Insight Engine/Data/preprocessed_article_df.csv"
preprocessed_article_df = pd.read_csv(path)

## Sentiment Analysis

In [ ]:
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
textblob_polarity = documents.apply(lambda x: TextBlob(x).sentiment[0])
vader_polarity = documents.apply(lambda x: SentimentIntensityAnalyzer().polarity_scores(x)['compound'])

In [ ]:
averaged_sentiment_score = (textblob_polarity + vader_polarity) / 2

In [ ]:
preprocessed_article_df['sentiment_score'] = averaged_sentiment_score

In [ ]:
from datetime import datetime
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import torch

## Retrieve and Re-rank

In [ ]:
def RetriveReRank(query, top_k=100):

  bi_encoder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
  bi_encoder.max_seq_length = 512     #Truncate long passages to 512 tokens
  top_k = top_k                          #Number of passages we want to retrieve with the bi-encoder

  question_embedding = bi_encoder.encode(query, convert_to_tensor=True)

  # corpus_embeddings = bi_encoder.encode(documents, convert_to_tensor=True, show_progress_bar=True)

  #The bi-encoder will retrieve 100 documents. We use a cross-encoder, to re-rank the results list to improve the quality
  cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')

  corpus_embeddings = np.load("/content/drive/MyDrive/Colab Notebooks/Technology News Insight Engine/Data/corpus_embeddings.npy")
  corpus_embeddings = torch.from_numpy(corpus_embeddings)

  query = "What is the initiatives of cloud accounting software platform?"
  question_embedding = bi_encoder.encode(query, convert_to_tensor=True)
  question_embedding = question_embedding.cuda()
  hits = util.semantic_search(question_embedding, corpus_embeddings, top_k=top_k)
  hits = hits[0]  # Get the hits for the first query

  ##### Re-Ranking #####
  # Now, score all retrieved passages with the cross_encoder
  cross_inp = [[query, documents[hit['corpus_id']]] for hit in hits]
  cross_scores = cross_encoder.predict(cross_inp)
  mean_socres = np.mean(cross_scores)

  # Sort results by the cross-encoder scores
  for idx in range(len(cross_scores)):
    hits[idx]['cross-score'] = cross_scores[idx]

  hits = sorted(hits, key=lambda x: x['score'], reverse=True)
  hits = sorted(hits, key=lambda x: x['cross-score'], reverse=True)

  candidates = pd.DataFrame(hits)

  sentiment_score = preprocessed_article_df.loc[candidates["corpus_id"]].sentiment_score.values
  published_at = pd.to_datetime(preprocessed_article_df.loc[candidates["corpus_id"]].published_at.values)

  candidates["sentiment_score"] = sentiment_score
  candidates["published_at"] = published_at

  candidates.sort_values(by=["published_at", "sentiment_score", "cross-score"],
                       ascending=[False, False, False])

  difference = (max(candidates["published_at"]) - min(candidates["published_at"])) / 2

  threshold = min(candidates["published_at"]) + difference

  latest_candidates = candidates[
      (candidates["published_at"] > threshold) &
       (candidates["sentiment_score"] > 0) &
        (candidates["cross-score"] > mean_socres)]

  selected_documents = documents[latest_candidates["corpus_id"].values]
  return selected_documents

## RAG

In [ ]:
from llama_index.core import VectorStoreIndex,SimpleDirectoryReader, PromptTemplate
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import BitsAndBytesConfig
from llama_index.core.response.notebook_utils import display_response
from llama_index.core import Settings
from langchain.embeddings import HuggingFaceEmbeddings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

In [ ]:
def messages_to_prompt(messages):
  prompt = ""
  for message in messages:
    if message.role == 'system':
      prompt += f"<|system|>\n{message.content}\n"
    elif message.role == 'user':
      prompt += f"<|user|>\n{message.content}\n"
    elif message.role == 'assistant':
      prompt += f"<|assistant|>\n{message.content}\n"
# ensure we start with a system prompt, insert blank if needed
  if not prompt.startswith("<|system|>\n"):
    prompt = "<|system|>\n\n" + prompt

  # add final assistant prompt
  prompt = prompt + "<|assistant|>\n"

  return prompt

In [ ]:
llm = HuggingFaceLLM(
    model_name="meta-llama/Llama-2-7b-chat-hf",
    tokenizer_name="meta-llama/Llama-2-7b-chat-hf",
    query_wrapper_prompt=PromptTemplate("<|system|>\n\n<|user|>\n{query_str}\n<|assistant|>\n"),
    context_window=3900,
    max_new_tokens=512,
    model_kwargs={"quantization_config": quantization_config},
    # tokenizer_kwargs={},
    generate_kwargs={"temperature": 0.3, "top_k": 50, "top_p": 0.95},
    messages_to_prompt=messages_to_prompt,
    device_map="auto",

)

In [ ]:
local_model_path = "BAAI/bge-small-en-v1.5"
embed_model = HuggingFaceEmbeddings(model_name=local_model_path)

In [ ]:
Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
Settings.num_output = 512
Settings.context_window = 3900

In [ ]:
RAG("What is the initiatives of cloud accounting software platform?")

Final Response: Cloud accounting software platforms are designed to provide a range of initiatives that help small businesses manage their finances more efficiently. Some of the key initiatives of cloud accounting software platforms include:

Automating payroll system: Cloud accounting software platforms automate the payroll system, eliminating the need for manual data entry and reducing the chances of errors.
Time-saving: Cloud accounting software platforms provide enormous time-saving benefits to the finance department, saving up to 10,000 hours annually.
Integration with other business systems: Cloud accounting software platforms integrate with other business systems such as CRM, HR, and inventory management systems, providing a seamless workflow.
Customizable dashboard: Cloud accounting software platforms provide a customizable dashboard that allows users to present important information at a glance, tailoring the system to their existing workflow.
Invoicing and project management: Cloud accounting software platforms provide invoicing and project management tools, allowing users to track projects and invoices, and send reminders to clients.
Tax management: Cloud accounting software platforms provide tax management tools that help users accurately estimate their tax liability and save money accordingly.
Subscription pricing: Cloud accounting software platforms offer subscription pricing plans that are flexible and scalable, allowing users to choose the plan that best suits their business needs.
Free trial: Cloud accounting software platforms offer a free trial, allowing users to test the solution before committing to a purchase.
Overall, cloud accounting software platforms are designed to streamline business operations, reduce manual data entry, and provide valuable insights to help small businesses make informed financial decisions.

In [ ]:
RAG("Who are the biggest providers of cloud accounting software platform and what type is being used?")

Final Response: Based on the context information provided, the biggest providers of cloud accounting software platforms are:

QuickBooks: QuickBooks is one of the most popular cloud accounting software platforms used by small businesses. It offers a variety of features such as invoicing, expense tracking, inventory management, and payroll processing.

Xero: Xero is another popular cloud accounting software platform that is widely used by small businesses. It offers features such as invoicing, expense tracking, and project management, as well as integration with other business tools such as payment gateways and banking apps.
FreshBooks: FreshBooks is a cloud accounting software platform that is specifically designed for freelancers and small businesses. It offers features such as invoicing, time tracking, and expense tracking, as well as integration with other business tools such as payment gateways and project management software.

Zoho Books: Zoho Books is a cloud accounting software platform that offers features such as invoicing, expense tracking, and project management. It also integrates with other Zoho business tools such as CRM and inventory management.
Wave: Wave is a cloud accounting software platform that is designed for small businesses and freelancers. It offers features such as invoicing, expense tracking, and payment processing, as well as integration with other business tools such as payment gateways and banking apps.

All of these platforms are cloud-based, which means they can be accessed from anywhere and are easy to use. They also offer various pricing plans, making them accessible to small businesses of all sizes.